# cognitive and affective outcomes analysis

relationship between VTA-ROI overlap and neuropsychological change after DBS.

**outcome per test**
- `RCI + PE` (practice-effect corrected reliable change index), primary when available (n ≥ 10)
- `SDS` (simple discrepancy score), used when RCI + PE is unavailable

**direction:** higher values = more improvement for all outcomes
- accuracy/performance scores: positive = improvement
- time/error scores: SDS negated so positive = faster/fewer errors
- mood/anxiety/apathy scales (BDI, BAI, PAS, SAS): SDS negated so positive = symptom reduction

**structure:** part 1 = full-sample correlations and regression, part 2 = non-zero overlap sensitivity

## 0. setup

In [ ]:
import os
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, pearsonr, spearmanr
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')

COG_FILE     = '<path to cog_tests.xlsx>'
OVERLAP_FILE = '<path to outcomes_data.xlsx>'
FIG_DIR      = 'figures_cog'
OUT_CSV      = 'cog_results_all.csv'

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')
os.makedirs(FIG_DIR, exist_ok=True)


# avg_GPi_premotor -> avg. GPi premotor
def clean_label(col):
    parts = col.split('_')
    return parts[0] + '. ' + ' '.join(parts[1:]) if len(parts) >= 2 else col


def save_fig(fig, filename):
    path = os.path.join(FIG_DIR, filename + '.tif')
    fig.savefig(path, format='tiff', dpi=300, bbox_inches='tight')
    print(f'  saved: {path}')

## 1. test registry

tests included, their primary outcome column, direction and domain. tests with n < 10 for the primary outcome are skipped.

In [ ]:
# (sheet_name, display_name, primary_outcome_col, higher_is_better, domain)
# higher_is_better=False -> outcome is negated so positive = improvement
TEST_REGISTRY = [
    # premorbid / global
    ('MoCA 7.2',                     'MoCA',                     'SDS',      True,  'Global Cognition'),
    ('Wechsler Test of Adult Reading','WTAR',                     'SDS',      True,  'Premorbid'),
    # processing speed / attention
    ('Oral TMT A Time',              'Oral TMT-A',               'SDS',      False, 'Processing Speed'),
    ('Oral TMT B Time',              'Oral TMT-B',               'SDS',      False, 'Processing Speed'),
    ('DKEFS Color Naming',           'DKEFS Color Naming',       'RCI + PE', True,  'Processing Speed'),
    ('DKEFS Word Reading',           'DKEFS Word Reading',       'RCI + PE', True,  'Processing Speed'),
    ('DKEFS Color Word Inhibition',  'DKEFS Inhibition',         'RCI + PE', True,  'Executive / Inhibition'),
    ('DKEFS Color Word Switching',   'DKEFS Switching',          'RCI + PE', True,  'Executive / Inhibition'),
    ('SDMT Oral',                    'SDMT Oral',                'RCI + PE', True,  'Processing Speed'),
    # working memory / attention
    ('NAB Digit Forward',            'Digit Span Forward',       'RCI + PE', True,  'Working Memory'),
    ('NAB Digit Backward',           'Digit Span Backward',      'RCI + PE', True,  'Working Memory'),
    ('NAB Num&Letters B Time',       'N&L-B Time',               'SDS',      False, 'Working Memory'),
    ('NAB Num&Letters B Accuracy',   'N&L-B Accuracy',           'SDS',      True,  'Working Memory'),
    ('NAB Num&Letters B Efficiency', 'N&L-B Efficiency',         'RCI + PE', True,  'Working Memory'),
    ('NAB Num&Letters C Time',       'N&L-C Time',               'SDS',      False, 'Working Memory'),
    ('NAB Num&Letters C Accuracy',   'N&L-C Accuracy',           'SDS',      True,  'Working Memory'),
    ('NAB Num&Letters C Efficiency', 'N&L-C Efficiency',         'RCI + PE', True,  'Working Memory'),
    # learning and memory
    ('NAB List Learning A',          'List Learning (A)',        'RCI + PE', True,  'Learning & Memory'),
    ('NAB List Learning Short Delay','List Learning Short Delay','RCI + PE', True,  'Learning & Memory'),
    ('NAB List Learning Long Delay', 'List Learning Long Delay', 'RCI + PE', True,  'Learning & Memory'),
    ('NAB Story Recall Immediate',   'Story Recall Immediate',   'RCI + PE', True,  'Learning & Memory'),
    ('NAB Story Recall Delay',       'Story Recall Delay',       'RCI + PE', True,  'Learning & Memory'),
    # language
    ('Letter Fluency (FAS)',         'Letter Fluency (FAS)',     'RCI + PE', True,  'Language'),
    ('Boston Naming Test',           'Boston Naming',            'RCI + PE', True,  'Language'),
    ('NAB Visual Discrimination',    'Visual Discrimination',    'RCI + PE', True,  'Visuospatial'),
    # mood / affective (lower raw = better, SDS negated)
    ('BDI',  'BDI (Depression)',  'SDS', False, 'Mood / Affective'),
    ('BAI',  'BAI (Anxiety)',     'SDS', False, 'Mood / Affective'),
    ('PAS',  'PAS (Anxiety)',     'SDS', False, 'Mood / Affective'),
    ('SAS',  'SAS (Apathy)',      'SDS', False, 'Mood / Affective'),
]

# lookup dicts
TEST_PRIMARY   = {s: col  for s, _, col, _, _ in TEST_REGISTRY}
TEST_DIRECTION = {s: hib  for s, _, _, hib, _ in TEST_REGISTRY}
TEST_LABEL     = {s: lbl  for s, lbl, _, _, _ in TEST_REGISTRY}
TEST_DOMAIN    = {s: dom  for s, _, _, _, dom  in TEST_REGISTRY}
ALL_SHEETS     = [s for s, *_ in TEST_REGISTRY]

print(f'Test registry: {len(TEST_REGISTRY)} tests')
print('\nDomain breakdown:')
for dom, cnt in Counter(TEST_DOMAIN[s] for s in ALL_SHEETS).items():
    print(f'  {dom}: {cnt} tests')

## 2. load cognitive data

In [ ]:
# sheet -> df with columns [id, target, sex, age, lead_num, outcome]
cog_data = {}

xl = pd.ExcelFile(COG_FILE)

for sheet, label, col, higher_is_better, domain in TEST_REGISTRY:
    if sheet not in xl.sheet_names:
        print(f'  sheet not found: {sheet}')
        continue
    df = pd.read_excel(xl, sheet_name=sheet)
    if col not in df.columns or df[col].notna().sum() < 10:
        print(f'  SKIP (n<10): {sheet}')
        continue
    df = df[['id','target','sex','age','lead_num', col]].copy()
    df = df.rename(columns={col: 'outcome'})
    # negate if lower = better
    if not higher_is_better:
        df['outcome'] = -df['outcome']
    df['sheet']   = sheet
    df['label']   = label
    df['domain']  = domain
    df['outcome_col'] = col
    cog_data[sheet] = df

print(f'Loaded {len(cog_data)} test sheets.')
print('\nSample sizes by test:')
for sheet, df in cog_data.items():
    n_stn = df[df['target']=='STN']['outcome'].notna().sum()
    n_gpi = df[df['target']=='GPi']['outcome'].notna().sum()
    col   = df['outcome_col'].iloc[0]
    print(f'  {TEST_LABEL[sheet]:<35} n={df["outcome"].notna().sum():>3} (STN={n_stn}, GPi={n_gpi})  [{col}]')

## 3. load and merge VTA overlap data

In [ ]:
df_overlap = pd.read_excel(OVERLAP_FILE)
print(f'Overlap data shape: {df_overlap.shape}')
print(f'Target distribution: {df_overlap["Target"].value_counts().to_dict()}')

# predictor columns (L_, R_, avg_ prefix)
overlap_cols = [c for c in df_overlap.columns
                if c.startswith(('L_', 'R_', 'avg_'))]
print(f'\nTotal overlap predictor columns: {len(overlap_cols)}')

# STN / GPi predictor splits
STN_PREDICTORS_L   = [c for c in overlap_cols if c.startswith('L_STN')]
STN_PREDICTORS_R   = [c for c in overlap_cols if c.startswith('R_STN')]
STN_PREDICTORS_AVG = [c for c in overlap_cols if c.startswith('avg_STN')]
GPi_PREDICTORS_L   = [c for c in overlap_cols if c.startswith('L_GPi') or c.startswith('L_GPe')]
GPi_PREDICTORS_R   = [c for c in overlap_cols if c.startswith('R_GPi') or c.startswith('R_GPe')]
GPi_PREDICTORS_AVG = [c for c in overlap_cols if c.startswith('avg_GPi') or c.startswith('avg_GPe')]

stn_preds_all = list(dict.fromkeys(STN_PREDICTORS_L + STN_PREDICTORS_R + STN_PREDICTORS_AVG))
gpi_preds_all = list(dict.fromkeys(GPi_PREDICTORS_L + GPi_PREDICTORS_R + GPi_PREDICTORS_AVG))
print(f'STN predictors: {len(stn_preds_all)}, GPi predictors: {len(gpi_preds_all)}')

In [ ]:
# merge each cog sheet with overlap data
id_col_overlap = 'id' if 'id' in df_overlap.columns else df_overlap.columns[0]

cog_merged = {}
for sheet, df_cog in cog_data.items():
    merged = df_cog.merge(
        df_overlap[[id_col_overlap, 'Laterality'] + overlap_cols],
        left_on='id', right_on=id_col_overlap, how='left'
    )
    cog_merged[sheet] = merged

## 4. descriptive statistics

In [ ]:
print('=== COHORT DEMOGRAPHICS (cognitive subsample) ===')
# largest test sheet as representative
ref_sheet = max(cog_merged, key=lambda s: cog_merged[s]['outcome'].notna().sum())
df_ref = cog_merged[ref_sheet].dropna(subset=['outcome'])

for target in ['STN', 'GPi']:
    sub = df_ref[df_ref['target'] == target]
    print(f'\n--- {target} (n={len(sub)}) ---')
    print(f'  Age : {sub["age"].mean():.1f} \u00b1 {sub["age"].std():.1f}')
    print(f'  Sex : {sub["sex"].value_counts().to_dict()}')
    print(f'  Lead: {sub["lead_num"].value_counts().to_dict()}')

print('\n=== OUTCOME DESCRIPTIVES BY TEST & TARGET ===')
rows = []
for sheet, df in cog_merged.items():
    for target in ['STN', 'GPi', 'ALL']:
        sub = df if target == 'ALL' else df[df['target'] == target]
        vals = sub['outcome'].dropna()
        if len(vals) < 3:
            continue
        rows.append({
            'Test': TEST_LABEL[sheet], 'Domain': TEST_DOMAIN[sheet],
            'Outcome': df['outcome_col'].iloc[0],
            'Target': target, 'n': len(vals),
            'Mean': round(vals.mean(), 3), 'SD': round(vals.std(), 3),
            'Median': round(vals.median(), 3),
            'Min': round(vals.min(), 3), 'Max': round(vals.max(), 3),
            '% improved': round((vals > 0).mean() * 100, 1)
        })
desc_df = pd.DataFrame(rows)
print(desc_df[desc_df['Target']=='ALL'].drop('Target',axis=1).reset_index(drop=True))

## 5. normality testing

In [ ]:
# outcome normality per test
print('=== Outcome normality by test ===')
for sheet, df in cog_merged.items():
    vals = df['outcome'].dropna()
    if len(vals) < 5: continue
    stat, p = shapiro(vals)
    flag = '  [non-normal]' if p <= 0.05 else ''
    print(f'  {TEST_LABEL[sheet]:<35} W={stat:.4f}  p={p:.4f}{flag}')

## 6. correlation analysis

Pearson and Spearman for each test × predictor pair. Spearman is primary given predictor non-normality. p-values are uncorrected (exploratory).

In [ ]:
def compute_correlations_cog(df_merged, predictors, target_filter=None, alpha=0.05):
    # target_filter: 'STN', 'GPi' or None (all)
    if target_filter:
        df = df_merged[df_merged['target'] == target_filter].copy()
    else:
        df = df_merged.copy()

    rows = []
    for pred in predictors:
        if pred not in df.columns: continue
        combined = df[['outcome', pred]].dropna()
        if len(combined) < 8: continue
        r_p, p_p = pearsonr(combined['outcome'], combined[pred])
        r_s, p_s = spearmanr(combined['outcome'], combined[pred])
        rows.append({
            'Predictor': pred, 'n': len(combined),
            'Pearson r': round(r_p, 3), 'p (Pearson)': round(p_p, 4),
            'Spearman rho': round(r_s, 3), 'p (Spearman)': round(p_s, 4),
            'sig_pearson':  '*' if p_p < alpha else '',
            'sig_spearman': '*' if p_s < alpha else ''
        })
    return pd.DataFrame(rows)


def run_all_correlations(target_filter=None, label='ALL'):
    all_sig = []
    for sheet, df in cog_merged.items():
        # target-appropriate predictors
        if target_filter == 'STN':
            preds = stn_preds_all
        elif target_filter == 'GPi':
            preds = gpi_preds_all
        else:
            preds = stn_preds_all + gpi_preds_all

        corr_df = compute_correlations_cog(df, preds, target_filter)
        if corr_df.empty: continue
        sig = corr_df[(corr_df['sig_pearson']=='*') | (corr_df['sig_spearman']=='*')].copy()
        if len(sig):
            sig.insert(0, 'Test', TEST_LABEL[sheet])
            sig.insert(1, 'Domain', TEST_DOMAIN[sheet])
            all_sig.append(sig)

    if all_sig:
        result = pd.concat(all_sig).reset_index(drop=True)
        print(f'\n=== Significant correlations [{label}] (p<0.05 Pearson or Spearman) ===')
        print(result)
        return result
    else:
        print(f'No significant correlations [{label}]')
        return pd.DataFrame()

sig_STN = run_all_correlations('STN', 'STN')
sig_GPi = run_all_correlations('GPi', 'GPi')

## 7. scatter plots of significant correlations

In [ ]:
def plot_sig_scatters(sig_df, target_filter, method='Spearman', p_thresh=0.05):
    if sig_df.empty:
        print('No significant results to plot.')
        return

    method_col = 'Spearman rho' if method == 'Spearman' else 'Pearson r'
    p_col      = 'p (Spearman)' if method == 'Spearman' else 'p (Pearson)'

    rows_to_plot = sig_df[sig_df[p_col] < p_thresh].copy()
    if rows_to_plot.empty:
        print(f'No {method} significant pairs.')
        return

    color = '#2196F3' if target_filter == 'STN' else '#FF9800'
    ncols = min(4, len(rows_to_plot))
    nrows = int(np.ceil(len(rows_to_plot) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
    axes = np.array(axes).flatten() if len(rows_to_plot) > 1 else [axes]

    for i, (_, row) in enumerate(rows_to_plot.iterrows()):
        ax = axes[i]
        sheet = [s for s, l, *_ in TEST_REGISTRY if l == row['Test']]
        if not sheet: continue
        df = cog_merged[sheet[0]]
        if target_filter:
            df = df[df['target'] == target_filter]
        combined = df[['outcome', row['Predictor']]].dropna()
        if len(combined) < 5: continue

        ax.scatter(combined[row['Predictor']], combined['outcome'],
                   alpha=0.7, color=color, edgecolors='white', s=55)
        m, b = np.polyfit(combined[row['Predictor']], combined['outcome'], 1)
        x_line = np.linspace(combined[row['Predictor']].min(),
                             combined[row['Predictor']].max(), 100)
        ax.plot(x_line, m*x_line+b, 'r--', linewidth=1.5)
        ax.axhline(0, color='gray', linestyle=':', linewidth=1, alpha=0.6)

        r_val = row[method_col]
        p_val = row[p_col]
        ax.set_xlabel(clean_label(row['Predictor']), fontsize=8)
        ax.set_ylabel(row['Test'], fontsize=8)
        ax.set_title(
            f"{row['Test']}\n~ {clean_label(row['Predictor'])}\n"            f"{method} r={r_val:.3f}, p={p_val:.4f} [n={len(combined)}]",
            fontsize=8, fontweight='bold'
        )

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f'{target_filter} Significant Correlations ({method}, p<{p_thresh})',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    save_fig(fig, f'scatter_cog_{target_filter}_{method}')
    plt.show()

plot_sig_scatters(sig_STN, 'STN', method='Spearman')
plot_sig_scatters(sig_STN, 'STN', method='Pearson')
plot_sig_scatters(sig_GPi, 'GPi', method='Spearman')
plot_sig_scatters(sig_GPi, 'GPi', method='Pearson')

## 8. linear regression, unadjusted and adjusted (age + sex + laterality)

In [ ]:
def regression_cog(df_merged, predictors, target_filter=None,
                   adjusted=False, alpha=0.05):
    if target_filter:
        df = df_merged[df_merged['target'] == target_filter].copy()
    else:
        df = df_merged.copy()

    if df['sex'].dtype == object:
        df['Sex_bin'] = (df['sex'].str.upper().str.strip() == 'M').astype(int)
    else:
        df['Sex_bin'] = df['sex']
    df['Lat_bin'] = (df['Laterality'].str.upper().str.strip() == 'BI').astype(int)

    rows = []
    for pred in predictors:
        if pred not in df.columns: continue
        needed = [pred, 'outcome', 'age', 'Sex_bin', 'Lat_bin'] if adjusted else [pred, 'outcome']
        combined = df[needed].dropna()
        if len(combined) < 8: continue
        try:
            cov = '+ age + Sex_bin + Lat_bin' if adjusted else ''
            model = smf.ols(f'outcome ~ Q("{pred}") {cov}', data=combined).fit()
            coef = model.params[f'Q("{pred}")']
            ci = model.conf_int().loc[f'Q("{pred}")'].values
            p = model.pvalues[f'Q("{pred}")']
            rows.append({
                'Predictor': pred,
                'beta': round(coef, 4),
                'CI_lo': round(ci[0], 4), 'CI_hi': round(ci[1], 4),
                'R2': round(model.rsquared, 3),
                'p': round(p, 4), 'sig': '*' if p < alpha else '',
                'n': len(combined)
            })
        except Exception:
            continue
    return pd.DataFrame(rows)


def run_all_regressions(target_filter, adjusted=False):
    # returns all results, not just significant
    adj_label = 'Adjusted' if adjusted else 'Unadjusted'
    all_rows = []
    preds = stn_preds_all if target_filter == 'STN' else gpi_preds_all

    for sheet, df in cog_merged.items():
        reg_df = regression_cog(df, preds, target_filter, adjusted)
        if reg_df.empty: continue
        reg_df = reg_df.copy()
        reg_df.insert(0, 'Test', TEST_LABEL[sheet])
        reg_df.insert(1, 'Domain', TEST_DOMAIN[sheet])
        all_rows.append(reg_df)

    if all_rows:
        result = pd.concat(all_rows).reset_index(drop=True)
        sig = result[result['sig'] == '*']
        print(f'\n=== Significant regressions [{target_filter}, {adj_label}] ===')
        if len(sig):
            print(sig[['Test','Domain','Predictor','beta','CI_lo','CI_hi','R2','p','n']])
        else:
            print('  None')
        print(f'\n  Total rows (all predictors): {len(result)}  |  Significant: {len(sig)}')
        return result
    else:
        print(f'No regressions ran [{target_filter}, {adj_label}]')
        return pd.DataFrame()

reg_STN_unadj = run_all_regressions('STN', adjusted=False)
reg_STN_adj   = run_all_regressions('STN', adjusted=True)
reg_GPi_unadj = run_all_regressions('GPi', adjusted=False)
reg_GPi_adj   = run_all_regressions('GPi', adjusted=True)

## 9. summary heatmap

Spearman ρ for tests and predictors with at least one significant result.

In [ ]:
def summary_heatmap(target_filter, sig_result_df, p_thresh=0.05):
    if sig_result_df.empty:
        print(f'No significant results for {target_filter}.')
        return

    # rows = tests, cols = predictors with at least one significant result
    sig_preds = sig_result_df['Predictor'].unique()
    sig_tests  = sig_result_df['Test'].unique()
    if len(sig_preds) == 0 or len(sig_tests) == 0:
        return

    r_matrix = pd.DataFrame(index=sig_tests, columns=sig_preds, dtype=float)
    p_matrix = pd.DataFrame(index=sig_tests, columns=sig_preds, dtype=float)

    for sheet, df in cog_merged.items():
        label = TEST_LABEL[sheet]
        if label not in sig_tests: continue
        sub = df[df['target']==target_filter] if target_filter else df
        for pred in sig_preds:
            if pred not in sub.columns: continue
            combined = sub[['outcome', pred]].dropna()
            if len(combined) < 8: continue
            r, p = spearmanr(combined['outcome'], combined[pred])
            r_matrix.loc[label, pred] = r
            p_matrix.loc[label, pred] = p

    r_matrix = r_matrix.dropna(how='all').dropna(axis=1, how='all').astype(float)
    p_matrix = p_matrix.loc[r_matrix.index, r_matrix.columns].astype(float)

    # r value + star
    annot = r_matrix.copy().astype(str)
    for row in r_matrix.index:
        for col in r_matrix.columns:
            r = r_matrix.loc[row, col]
            p = p_matrix.loc[row, col]
            if pd.isna(r):
                annot.loc[row, col] = ''
            else:
                star = '*' if p < p_thresh else ''
                annot.loc[row, col] = f'{r:.2f}{star}'

    col_labels = [clean_label(c) for c in r_matrix.columns]

    fig, ax = plt.subplots(figsize=(max(6, len(r_matrix.columns)*1.1),
                                    max(4, len(r_matrix)*0.6)))
    sns.heatmap(r_matrix, annot=annot, fmt='', cmap='RdBu', center=0,
                vmin=-0.6, vmax=0.6, linewidths=0.4,
                xticklabels=col_labels, yticklabels=r_matrix.index,
                ax=ax, annot_kws={'size': 8})
    ax.set_xticklabels(col_labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(r_matrix.index, fontsize=8)
    ax.set_title(
        f'{target_filter} Spearman ρ: Cognitive Outcomes × VTA Overlap\n* p<0.05',
        fontweight='bold', fontsize=11
    )
    plt.tight_layout()
    save_fig(fig, f'heatmap_cog_{target_filter}')
    plt.show()

summary_heatmap('STN', sig_STN)
summary_heatmap('GPi', sig_GPi)

## 10. STN vs GPi by test

In [ ]:
print('=== STN vs GPi: Cognitive outcome comparisons ===')
comparison_rows = []
for sheet, df in cog_merged.items():
    stn_vals = df[df['target']=='STN']['outcome'].dropna()
    gpi_vals = df[df['target']=='GPi']['outcome'].dropna()
    if len(stn_vals) < 5 or len(gpi_vals) < 5: continue
    stat, p = stats.mannwhitneyu(stn_vals, gpi_vals, alternative='two-sided')
    r_rb = 1 - (2*stat)/(len(stn_vals)*len(gpi_vals))
    comparison_rows.append({
        'Test': TEST_LABEL[sheet], 'Domain': TEST_DOMAIN[sheet],
        'Outcome': df['outcome_col'].iloc[0],
        'n_STN': len(stn_vals), 'n_GPi': len(gpi_vals),
        'Median_STN': round(stn_vals.median(),3),
        'Median_GPi': round(gpi_vals.median(),3),
        'p': round(p,4), 'r_rb': round(r_rb,3),
        'sig': '*' if p < 0.05 else ''
    })

comp_df = pd.DataFrame(comparison_rows).sort_values('p')
print('Significant STN vs GPi differences:')
print(comp_df[comp_df['sig']=='*'].reset_index(drop=True))
print('\nFull comparison table:')
print(comp_df.reset_index(drop=True))

# part 2. non-zero overlap sensitivity analyses

same as part 1, excluding patients with zero overlap for each predictor. separates dose-response (both) from overlap vs no-overlap contrasts (full sample only).

In [ ]:
def nonzero_subset_cog(df, pred_col):
    return df[df[pred_col].notna() & (df[pred_col] > 0)]

# zero-overlap % in the cognitive subsample, predictors > 15% zeros
print('=== Zero overlap % in cognitive subsample ===')
ref_df = list(cog_merged.values())[0]
for pred in stn_preds_all + gpi_preds_all:
    if pred not in ref_df.columns: continue
    n_zero  = (ref_df[pred] == 0).sum()
    n_total = ref_df[pred].notna().sum()
    if n_total == 0: continue
    pct = 100*n_zero/n_total
    if pct > 15:
        print(f'  {pred:<35} {n_zero}/{n_total} zeros ({pct:.0f}%)')

In [ ]:
def compute_correlations_nonzero_cog(df_merged, predictors, target_filter=None, alpha=0.05):
    if target_filter:
        df_base = df_merged[df_merged['target'] == target_filter].copy()
    else:
        df_base = df_merged.copy()

    rows = []
    for pred in predictors:
        if pred not in df_base.columns: continue
        nz = nonzero_subset_cog(df_base, pred)
        combined = nz[['outcome', pred]].dropna()
        if len(combined) < 8: continue
        r_p, p_p = pearsonr(combined['outcome'], combined[pred])
        r_s, p_s = spearmanr(combined['outcome'], combined[pred])
        rows.append({
            'Predictor': pred, 'n (nonzero)': len(combined),
            'Pearson r': round(r_p, 3), 'p (Pearson)': round(p_p, 4),
            'Spearman rho': round(r_s, 3), 'p (Spearman)': round(p_s, 4),
            'sig_pearson':  '*' if p_p < alpha else '',
            'sig_spearman': '*' if p_s < alpha else ''
        })
    return pd.DataFrame(rows)


def run_all_corr_nonzero(target_filter, label):
    all_sig = []
    preds = stn_preds_all if target_filter == 'STN' else gpi_preds_all
    for sheet, df in cog_merged.items():
        corr_df = compute_correlations_nonzero_cog(df, preds, target_filter)
        if corr_df.empty: continue
        sig = corr_df[(corr_df['sig_pearson']=='*')|(corr_df['sig_spearman']=='*')].copy()
        if len(sig):
            sig.insert(0, 'Test', TEST_LABEL[sheet])
            sig.insert(1, 'Domain', TEST_DOMAIN[sheet])
            all_sig.append(sig)
    if all_sig:
        result = pd.concat(all_sig).reset_index(drop=True)
        print(f'\n=== Non-zero significant correlations [{label}] ===')
        print(result)
        return result
    else:
        print(f'No significant non-zero correlations [{label}]')
        return pd.DataFrame()

nz_sig_STN = run_all_corr_nonzero('STN', 'STN')
nz_sig_GPi = run_all_corr_nonzero('GPi', 'GPi')

In [ ]:
def run_all_reg_nonzero(target_filter, adjusted=False):
    adj_label = 'Adjusted' if adjusted else 'Unadjusted'
    all_rows = []
    preds = stn_preds_all if target_filter == 'STN' else gpi_preds_all

    for sheet, df in cog_merged.items():
        if target_filter:
            df_sub = df[df['target'] == target_filter].copy()
        else:
            df_sub = df.copy()

        if df_sub['sex'].dtype == object:
            df_sub['Sex_bin'] = (df_sub['sex'].str.upper().str.strip() == 'M').astype(int)
        else:
            df_sub['Sex_bin'] = df_sub['sex']
        df_sub['Lat_bin'] = (df_sub['Laterality'].str.upper().str.strip() == 'BI').astype(int)

        rows = []
        for pred in preds:
            if pred not in df_sub.columns: continue
            nz = nonzero_subset_cog(df_sub, pred)
            needed = [pred,'outcome','age','Sex_bin'] if adjusted else [pred,'outcome']
            combined = nz[needed].dropna()
            if len(combined) < 8: continue
            try:
                cov = '+ age + Sex_bin' if adjusted else ''
                model = smf.ols(f'outcome ~ Q("{pred}") {cov}', data=combined).fit()
                coef = model.params[f'Q("{pred}")']
                ci = model.conf_int().loc[f'Q("{pred}")'].values
                p = model.pvalues[f'Q("{pred}")']
                rows.append({
                    'Predictor': pred,
                    'beta': round(coef,4), 'CI_lo': round(ci[0],4),
                    'CI_hi': round(ci[1],4), 'R2': round(model.rsquared,3),
                    'p': round(p,4), 'sig': '*' if p < 0.05 else '',
                    'n (nonzero)': len(combined)
                })
            except Exception:
                continue

        if rows:
            reg_df = pd.DataFrame(rows)
            reg_df.insert(0,'Test', TEST_LABEL[sheet])
            reg_df.insert(1,'Domain', TEST_DOMAIN[sheet])
            all_rows.append(reg_df)

    if all_rows:
        result = pd.concat(all_rows).reset_index(drop=True)
        sig = result[result['sig']=='*']
        print(f'\n=== Non-zero significant regressions [{target_filter}, {adj_label}] ===')
        if len(sig):
            print(sig[['Test','Domain','Predictor','beta','CI_lo','CI_hi','R2','p','n (nonzero)']])
        else:
            print('  None')
        print(f'\n  Total rows (all predictors): {len(result)}  |  Significant: {len(sig)}')
        return result
    else:
        print(f'No non-zero regressions ran [{target_filter}, {adj_label}]')
        return pd.DataFrame()

nz_reg_STN_unadj = run_all_reg_nonzero('STN', adjusted=False)
nz_reg_STN_adj   = run_all_reg_nonzero('STN', adjusted=True)
nz_reg_GPi_unadj = run_all_reg_nonzero('GPi', adjusted=False)
nz_reg_GPi_adj   = run_all_reg_nonzero('GPi', adjusted=True)

## 11. full sample vs non-zero

In [ ]:
def compare_full_vs_nonzero(full_df, nz_df, label):
    if full_df is None or full_df.empty:
        print(f'{label}: no full-sample significant results.')
        return
    full_sig = set(zip(full_df['Test'], full_df['Predictor']))
    nz_sig   = set(zip(nz_df['Test'],  nz_df['Predictor'])) if nz_df is not None and not nz_df.empty else set()

    both       = full_sig & nz_sig
    full_only  = full_sig - nz_sig
    nz_only    = nz_sig   - full_sig

    print(f'\n--- {label} ---')
    print(f'  Consistent (dose-response)    : {sorted(both) or "None"}')
    print(f'  Full-sample only (binary?)    : {sorted(full_only) or "None"}')
    print(f'  Non-zero only (zeros suppress): {sorted(nz_only) or "None"}')

print('FULL SAMPLE vs NON-ZERO, CORRELATIONS')
compare_full_vs_nonzero(sig_STN, nz_sig_STN, 'STN Spearman/Pearson')
compare_full_vs_nonzero(sig_GPi, nz_sig_GPi, 'GPi Spearman/Pearson')

print('\nFULL SAMPLE vs NON-ZERO, REGRESSION (ADJUSTED)')
compare_full_vs_nonzero(reg_STN_adj, nz_reg_STN_adj, 'STN Adjusted')
compare_full_vs_nonzero(reg_GPi_adj, nz_reg_GPi_adj, 'GPi Adjusted')

## 12. summary of all significant results

In [ ]:
print('ALL SIGNIFICANT COGNITIVE FINDINGS (p<0.05)')

for label, full_df, nz_df in [
    ('STN Correlations',             sig_STN,       nz_sig_STN),
    ('GPi Correlations',             sig_GPi,       nz_sig_GPi),
    ('STN Regression (unadj)',       reg_STN_unadj, nz_reg_STN_unadj),
    ('STN Regression (adj)',         reg_STN_adj,   nz_reg_STN_adj),
    ('GPi Regression (unadj)',       reg_GPi_unadj, nz_reg_GPi_unadj),
    ('GPi Regression (adj)',         reg_GPi_adj,   nz_reg_GPi_adj),
]:
    print(f'\n--- {label} ---')
    if full_df is None or full_df.empty:
        print('  None')
        continue
    cols = [c for c in ['Test','Domain','Predictor','Spearman rho','p (Spearman)',
                         'Pearson r','p (Pearson)','beta','p','n','n (nonzero)']
            if c in full_df.columns]
    print(full_df[cols].sort_values(['Domain','Test','p' if 'p' in full_df.columns else 'p (Spearman)'])
            .reset_index(drop=True))

print('\n* = p<0.05 (uncorrected, exploratory)')

## notes on interpretation

- **outcome direction**: all outcomes coded so positive = improvement (time/error and mood/symptom SDS negated).
- **primary method**: Spearman (pre-specified given predictor non-normality), Pearson also reported.
- **full sample vs non-zero**: significant in both = dose-response. full sample only = likely overlap vs no-overlap contrast. non-zero only = signal masked by zero-overlap patients.
- **multiple comparisons**: p-values are uncorrected. with 29 tests × multiple predictors, ~5% of results are expected by chance.
- **sample sizes**: n varies per test and predictor and is reported in all outputs.
- **figures**: saved as 300 dpi .tif to `figures_cog/`.

## export results

In [ ]:
def tag(df, analysis, sample):
    d = df.copy()
    d.insert(0, 'Sample', sample)
    d.insert(0, 'Analysis', analysis)
    return d

frames = []

for analysis, df_res in [
    ('Regression_STN_Unadjusted', reg_STN_unadj),
    ('Regression_STN_Adjusted',   reg_STN_adj),
    ('Regression_GPi_Unadjusted', reg_GPi_unadj),
    ('Regression_GPi_Adjusted',   reg_GPi_adj),
    ('Regression_NZ_STN_Unadjusted', nz_reg_STN_unadj),
    ('Regression_NZ_STN_Adjusted',   nz_reg_STN_adj),
    ('Regression_NZ_GPi_Unadjusted', nz_reg_GPi_unadj),
    ('Regression_NZ_GPi_Adjusted',   nz_reg_GPi_adj),
]:
    if df_res is not None and not df_res.empty:
        sample = 'full' if '_NZ_' not in analysis else 'nonzero'
        frames.append(tag(df_res, analysis, sample))

out = pd.concat(frames, ignore_index=True, sort=False)
out.to_csv(OUT_CSV, index=False)
print(f'saved {len(out)} rows to {OUT_CSV}')